# Taller 1. Fundamentos de modelación

| Campo | Valor |
|---|---|
| Asignatura | Modelación y Simulación Computacional (2026-2) |
| Unidad | Unidad 1. Fundamentos de modelación en ingeniería |
| Subtemas del plan | 1.1 a 1.5 |
| Instrumento | Taller 1, peso de 7.5 % del curso |
| Modalidad | Parejas |
| Docente | Daniel D. Otero Meza |

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/talleres/taller_1.ipynb)

Este cuaderno es el andamiaje del Taller 1. Usted resuelve las celdas marcadas con
`# COMPLETE` y ejecuta después la celda de verificación que sigue a cada una. Mientras
la bandera `REVISAR` valga `False`, las verificaciones informan sin detener el cuaderno,
de modo que puede recorrerlo completo desde el primer momento. Cuando haya terminado,
ponga `REVISAR = True`, reinicie el núcleo y ejecute todo de arriba abajo. La entrega es
válida solo si el cuaderno corre completo en ese estado.

## Objetivos de aprendizaje

Al terminar este taller usted debe ser capaz de

1. traducir un enunciado de ingeniería en una pregunta que un modelo pueda responder, con
   variable de interés, horizonte y tolerancia;
2. construir el modelo conceptual de un sistema, delimitar su frontera y registrar sus
   supuestos con la razón, el efecto y la prueba de cada uno;
3. clasificar un modelo según los seis criterios transversales de la Tabla 1.1 del libro y
   anticipar la consecuencia práctica de cada posición;
4. contar los grados de libertad de una formulación estacionaria antes de programarla y
   detectar ecuaciones redundantes;
5. adimensionalizar un balance, identificar los grupos que gobiernan la respuesta y
   comprobar numéricamente la invariancia que predicen;
6. delimitar el dominio de validez de un modelo y sostener con él una decisión de diseño.

## Puesta a punto

La primera celda detecta el entorno e instala solo lo que falte, de manera que el cuaderno
abra sin edición manual tanto en Google Colab como en JupyterLab.

In [ ]:
import importlib, subprocess, sys

EN_COLAB = "google.colab" in sys.modules

def asegurar(paquetes):
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)

asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})

In [ ]:
import hashlib
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy.interpolate import PchipInterpolator
from scipy.optimize import brentq

%matplotlib inline

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
          "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

REVISAR = False   # póngalo en True cuando haya completado todas las celdas


def comprobar(nombre, condicion, detalle=""):
    """Informa el resultado de una verificación sin detener el cuaderno."""
    marca = "ok     " if condicion else "revisar"
    print(f"[{marca}] {nombre}" + (f"  ->  {detalle}" if detalle else ""))
    if REVISAR:
        assert condicion, f"{nombre} no se cumple. {detalle}"
    return bool(condicion)


def huella(texto):
    """Huella corta de una respuesta escrita, sin tildes y sin mayúsculas."""
    t = unicodedata.normalize("NFKD", str(texto).strip().lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return hashlib.sha256(t.encode("utf-8")).hexdigest()[:10]


print(f"NumPy {np.__version__} | SciPy disponible | pandas {pd.__version__} "
      f"| SymPy {sp.__version__} | semilla {SEMILLA}")

## Contexto del problema

La Institución Educativa rural del corregimiento de Las Flores, en el municipio de Sincé,
departamento de Sucre, atiende a 180 estudiantes en jornada única. El acueducto veredal
entrega agua de manera intermitente y la institución complementa el suministro con
carrotanques, cuyo costo el rector considera insostenible. La Secretaría de Educación
estudia instalar un sistema de aprovechamiento de agua lluvia que cubra los usos no
potables, es decir, la descarga de las baterías sanitarias, el lavamanos y el aseo general.

La cubierta útil, medida en proyección horizontal, suma 650 m2 de teja de fibrocemento en
buen estado. El coeficiente de escorrentía adoptado es 0.85 y de cada evento se descarta
el primer milímetro por el dispositivo de lavado inicial. La demanda no potable estimada
asciende a 1.20 m3 por día, equivalente a 6.7 L por estudiante y por día. El anteproyecto
propone un tanque enterrado de 30 m3 y afirma que el sistema es viable porque la lluvia
media anual de la zona supera con holgura la demanda anual.

Usted debe examinar esa afirmación. El régimen de lluvia de la sabana de Sucre es
marcadamente estacional, con una temporada seca larga entre diciembre y marzo, y esa
estacionalidad no aparece en un balance anual. La pregunta que el taller resuelve es si el
tanque propuesto sostiene el servicio y, si no lo sostiene, qué volumen haría falta y a
qué costo.

In [ ]:
# Datos del caso. No modifique estos valores.
AREA_TECHO = 650.0        # m2 de cubierta en proyección horizontal
COEF_ESCORRENTIA = 0.85   # fracción de la lluvia que llega al tanque
LAVADO_INICIAL = 1.0      # mm de cada evento que descarta el dispositivo
DEMANDA = 1.20            # m3/dia de agua no potable
ESTUDIANTES = 180
VOLUMEN_BASE = 30.0       # m3 del tanque propuesto en el anteproyecto
ANIOS = 3                 # el primer año se descarta como calentamiento

CLIMA = {1: (0.06, 6.0), 2: (0.05, 6.0), 3: (0.08, 8.0), 4: (0.27, 11.0),
         5: (0.37, 12.0), 6: (0.35, 11.0), 7: (0.32, 10.0), 8: (0.36, 11.0),
         9: (0.42, 12.0), 10: (0.45, 13.0), 11: (0.30, 10.0), 12: (0.12, 7.0)}


def serie_lluvia(anios=ANIOS, semilla=SEMILLA):
    """Serie diaria sintética de lluvia, en mm, con el régimen de la sabana de Sucre.

    Cada mes aporta una probabilidad de día lluvioso y una lámina media
    exponencial en los días con lluvia, tomadas de CLIMA.
    """
    gen = np.random.default_rng(semilla)
    fechas = pd.date_range("2026-01-01", periods=365 * anios, freq="D")
    prob = np.array([CLIMA[m][0] for m in fechas.month])
    media = np.array([CLIMA[m][1] for m in fechas.month])
    moja = gen.random(fechas.size) < prob
    lamina = np.where(moja, gen.exponential(media), 0.0)
    return pd.Series(np.round(lamina, 1), index=fechas, name="lluvia_mm")


lluvia = serie_lluvia()
por_anio = pd.Series(
    [lluvia.to_numpy()[i * 365:(i + 1) * 365].sum() for i in range(ANIOS)],
    index=[f"año {i + 1}" for i in range(ANIOS)], name="lluvia_mm")
print(por_anio.round(1))
print(f"lluvia media anual {por_anio.mean():.1f} mm en {int((lluvia > 0).sum())} "
      f"días con lluvia de {ANIOS * 365} simulados")

## Tarea 1. Pregunta de ingeniería, modelo conceptual y registro de supuestos

La Definición 1.1 del libro exige que un modelo se formule con un propósito declarado, y la
Definición 1.3 fija el contenido del modelo conceptual, que es la frontera del sistema, los
componentes retenidos, los flujos y las hipótesis simplificadoras. El entregable de esta
etapa no es código sino un documento revisable por un tercero, con la estructura de cuatro
columnas de la Tabla 1.3 del libro.

Escriba en la celda de texto siguiente la pregunta de ingeniería del caso, con su variable
de interés, su horizonte, su resolución temporal y su tolerancia, y describa la frontera del
sistema indicando qué queda dentro, qué entra y qué se ignora. Complete después el registro
de supuestos con al menos cinco filas.

### Respuesta de la Tarea 1

**Pregunta de ingeniería.** COMPLETE. Redáctela en una sola oración, con la variable de
interés, el horizonte, la resolución y la tolerancia admisible.

**Frontera del sistema.** COMPLETE. Enumere qué componentes quedan dentro de la frontera,
qué magnitudes entran como forzamiento y qué procesos se dejan fuera de manera deliberada.

**Comentario.** COMPLETE. Explique por qué el enunciado del anteproyecto, que compara la
lluvia media anual con la demanda anual, no es todavía una pregunta de ingeniería.

In [ ]:
# COMPLETE: construya el registro de supuestos del modelo conceptual.
#           Debe tener al menos cinco filas y las cuatro columnas de la Tabla 1.3
#           del libro, es decir, el enunciado del supuesto, la razón por la cual se
#           adopta, el efecto esperado si resultara falso y la prueba con la que
#           podría contrastarse. Ninguna celda puede quedar vacía.
registro_supuestos = pd.DataFrame(
    [["", "", "", ""]],                      # fila de partida, evidentemente incompleta
    columns=["supuesto", "razon", "efecto_si_es_falso", "prueba"])

registro_supuestos

In [ ]:
col_ok = list(registro_supuestos.columns) == ["supuesto", "razon",
                                              "efecto_si_es_falso", "prueba"]
lleno = bool((registro_supuestos.map(lambda x: str(x).strip() != "")).all().all())
comprobar("el registro tiene las cuatro columnas de la Tabla 1.3", col_ok,
          f"columnas {list(registro_supuestos.columns)}")
comprobar("el registro tiene al menos cinco supuestos", len(registro_supuestos) >= 5,
          f"filas {len(registro_supuestos)}")
comprobar("ninguna celda del registro está vacía", lleno)

## Tarea 2. Clasificación del modelo según los seis criterios

La Tabla 1.1 del libro ordena los modelos con seis criterios transversales que se aplican de
forma simultánea, de suerte que un modelo no pertenece a una categoría sino que ocupa una
posición en cada uno de los seis ejes. Clasifique el modelo de balance diario del tanque tal
como se va a implementar en la Tarea 4, esto es, tomando la serie de lluvia como un dato de
entrada ya conocido.

Elija para cada criterio una de las opciones del diccionario `OPCIONES`, escrita exactamente
como aparece allí. La verificación compara la huella de su respuesta contra la huella de la
respuesta esperada, de modo que no revela la solución.

In [ ]:
OPCIONES = {
    "origen":       ("mecanicista", "empirico", "hibrido"),
    "tiempo":       ("estacionario", "dinamico"),
    "aleatoriedad": ("deterministico", "estocastico"),
    "espacio":      ("concentrados", "distribuidos"),
    "linealidad":   ("lineal", "no lineal"),
    "estado":       ("continuo", "discreto"),
}

# COMPLETE: escriba la posición del modelo en cada uno de los seis ejes.
clasificacion = {
    "origen": "empirico",             # valores de partida evidentemente incorrectos
    "tiempo": "estacionario",
    "aleatoriedad": "estocastico",
    "espacio": "distribuidos",
    "linealidad": "lineal",
    "estado": "discreto",
}

In [ ]:
ESPERADO = {'origen': 'b5076d82bd', 'tiempo': '2208e25424', 'aleatoriedad': '4e1e58df8c', 'espacio': '2eed25cbd0', 'linealidad': '223d5b929f', 'estado': '217fbff6d4'}

for eje, valor in clasificacion.items():
    valido = valor in OPCIONES[eje]
    acierta = valido and huella(valor) == ESPERADO[eje]
    comprobar(f"criterio de {eje}", acierta,
              "opción no admitida" if not valido else
              ("posición correcta" if acierta else "revise la Tabla 1.1 del libro"))

### Comentario de la Tarea 2

**Consecuencia de la clasificación.** COMPLETE. Explique en tres o cuatro oraciones qué
implica la posición obtenida sobre el método de solución que hará falta y sobre lo que el
modelo puede y no puede responder.

## Tarea 3. Forma canónica y grados de libertad de la versión estacionaria

La Ecuación 1.4 del libro escribe todo modelo dinámico separando el estado, las entradas
manipulables, las perturbaciones y los parámetros. Esa separación no es un formalismo, porque
el estatus de una magnitud determina qué se puede hacer con ella, según discute la Sección
1.4 del libro.

La Definición 1.10 introduce los grados de libertad de un modelo estacionario y el Listado 1.4
los cuenta por el rango de la matriz jacobiana. Aquí se usa una tolerancia relativa al mayor
elemento del jacobiano, porque las magnitudes del problema difieren en varios órdenes y una
tolerancia absoluta clasificaría mal el rango.

Aplique el conteo al modelo estacionario que el anteproyecto utiliza sin nombrarlo, que es el
balance anual de volúmenes.

### Forma canónica del modelo dinámico

**Estado.** COMPLETE. **Entrada manipulable.** COMPLETE. **Perturbación.** COMPLETE.
**Parámetros.** COMPLETE. **Salida de interés.** COMPLETE. **Condición inicial.** COMPLETE.

In [ ]:
def grados_libertad(residuos, punto, n_especificadas, paso=1e-4):
    """Grados de libertad y ecuaciones redundantes por el rango del jacobiano.

    Sigue la estrategia del Listado 1.4 del libro, con una tolerancia relativa
    al mayor elemento del jacobiano.
    """
    punto = np.asarray(punto, dtype=float)
    base = np.asarray(residuos(punto), dtype=float)
    jacobiano = np.empty((base.size, punto.size))
    for j in range(punto.size):
        desplazado = punto.copy()
        desplazado[j] += paso
        jacobiano[:, j] = (np.asarray(residuos(desplazado)) - base) / paso
    tol = 1e-8 * max(1.0, float(np.abs(jacobiano).max()))
    rango = int(np.linalg.matrix_rank(jacobiano, tol=tol))
    return punto.size - rango - n_especificadas, base.size - rango


# Orden de las incógnitas del modelo estacionario anual
# v = [Q_a, C, A, P_a, D_a, N, q, R_a]
#     Q_a  volumen anual captado (m3)      C    coeficiente de escorrentía
#     A    área de cubierta (m2)           P_a  lluvia anual (mm)
#     D_a  demanda anual (m3)              N    número de estudiantes
#     q    dotación (L por estudiante y día)  R_a  excedente vertido al año (m3)
PUNTO = np.array([558.0, 0.85, 650.0, 1010.0, 438.0, 180.0, 6.67, 120.0])

In [ ]:
# COMPLETE: escriba los tres residuos del modelo estacionario anual.
#           r1  el volumen anual captado sale del área, del coeficiente y de la lluvia
#           r2  la demanda anual sale del número de estudiantes y de la dotación
#           r3  el balance anual reparte lo captado entre lo consumido y el excedente
def residuos_anual(v):
    Q_a, C, A, P_a, D_a, N, q, R_a = v
    r1 = 0.0                     # valores de partida evidentemente incorrectos
    r2 = 0.0
    r3 = 0.0
    return np.array([r1, r2, r3])


# COMPLETE: agregue una cuarta ecuación que sea combinación lineal de las tres
#           anteriores, para comprobar que el conteo la detecta como redundante.
def residuos_anual_redundante(v):
    Q_a, C, A, P_a, D_a, N, q, R_a = v
    r4 = 0.0
    return np.concatenate([residuos_anual(v), [r4]])

In [ ]:
gl_libre, red_libre = grados_libertad(residuos_anual, PUNTO, 0)
gl_fijo, _ = grados_libertad(residuos_anual, PUNTO, 5)
gl_red, red_red = grados_libertad(residuos_anual_redundante, PUNTO, 0)

print(f"sin especificar nada  GL = {gl_libre}, ecuaciones redundantes = {red_libre}")
print(f"especificando C, A, P_a, N y q  GL = {gl_fijo}")
print(f"con la cuarta ecuación  GL = {gl_red}, ecuaciones redundantes = {red_red}")

comprobar("el sistema anual tiene cinco grados de libertad", gl_libre == 5,
          f"se obtuvo {gl_libre}")
comprobar("queda bien planteado al especificar cinco magnitudes", gl_fijo == 0,
          f"se obtuvo {gl_fijo}")
comprobar("las tres ecuaciones originales son independientes", red_libre == 0,
          f"redundantes {red_libre}")
comprobar("la cuarta ecuación se detecta como redundante", red_red == 1,
          f"redundantes {red_red}")
comprobar("los residuos se anulan en un punto admisible del modelo",
          float(np.abs(residuos_anual(PUNTO)).max()) < 1.0,
          f"residuo máximo {float(np.abs(residuos_anual(PUNTO)).max()):.3f}")

## Tarea 4. Balance diario del tanque y verificación del cierre

El modelo dinámico recorre la serie día a día, con la misma estructura del Listado 1.2 del
libro. El orden de las operaciones dentro del día es una decisión de modelación que hay que
declarar, y aquí se adopta el siguiente. Primero entra el volumen captado, después se vierte
por el rebosadero todo lo que exceda la capacidad del tanque, y por último se entrega la
demanda con lo que quede disponible.

El volumen captado en el día se obtiene de la lámina efectiva, que es la lluvia menos el
milímetro del lavado inicial y nunca negativa, multiplicada por el área de cubierta y por el
coeficiente de escorrentía. La conversión de milímetros por metro cuadrado a metros cúbicos
introduce el factor mil.

La verificación que acompaña a esta tarea es el cierre exacto del balance de masa, que debe
cumplirse hasta la precisión de la máquina. Un cierre distinto de cero delata un error de
programación antes que un fenómeno físico.

In [ ]:
def balance_diario(lluvia_mm, area, coef, lavado, volumen, demanda, inicial=0.0):
    """Balance diario del tanque de aprovechamiento.

    Devuelve cuatro arreglos en m3, con el volumen almacenado al final de cada día,
    el volumen captado, el volumen suministrado y el volumen rebosado.
    """
    n = int(np.asarray(lluvia_mm).size)
    almacenado = np.zeros(n)
    captado = np.zeros(n)
    suministrado = np.zeros(n)
    rebosado = np.zeros(n)
    s = float(inicial)
    for i in range(n):
        # COMPLETE: volumen captado del día i, en m3, a partir de la lámina efectiva
        q = 0.0
        # COMPLETE: acumule q en el tanque, calcule el rebose contra `volumen`,
        #           entregue hasta `demanda` con lo que quede y actualice s
        r = 0.0
        d = 0.0
        almacenado[i], captado[i] = s, q
        suministrado[i], rebosado[i] = d, r
    return almacenado, captado, suministrado, rebosado

In [ ]:
alm, cap, sumin, reb = balance_diario(
    lluvia.to_numpy(), AREA_TECHO, COEF_ESCORRENTIA, LAVADO_INICIAL,
    VOLUMEN_BASE, DEMANDA)

cierre = alm[-1] - 0.0 - (cap.sum() - sumin.sum() - reb.sum())
comprobar("el balance de masa cierra", abs(cierre) < 1e-9,
          f"residuo {cierre:.3e} m3 sobre {cap.sum():.1f} m3 captados")
comprobar("el almacenamiento nunca supera la capacidad",
          float(alm.max()) <= VOLUMEN_BASE + 1e-12, f"máximo {float(alm.max()):.3f} m3")
comprobar("el almacenamiento nunca es negativo", float(alm.min()) >= -1e-12,
          f"mínimo {float(alm.min()):.3f} m3")
comprobar("nunca se entrega más que la demanda diaria",
          float(sumin.max()) <= DEMANDA + 1e-12, f"máximo {float(sumin.max()):.3f} m3")
comprobar("el sistema capta agua durante la temporada de lluvias",
          float(cap.sum()) > 0.0, f"captado {float(cap.sum()):.1f} m3 en {ANIOS} años")

In [ ]:
EVAL = slice(365, 365 * ANIOS)     # se descarta el primer año de calentamiento
n_eval = 365 * (ANIOS - 1)

oferta_anual = cap[EVAL].sum() / (ANIOS - 1)
demanda_anual = DEMANDA * 365.0
confiabilidad_base = float(np.mean(sumin[EVAL] >= DEMANDA - 1e-12))
deficit_anual = (demanda_anual * (ANIOS - 1) - sumin[EVAL].sum()) / (ANIOS - 1)

print(f"oferta media anual        {oferta_anual:8.1f} m3")
print(f"demanda anual             {demanda_anual:8.1f} m3")
print(f"razón oferta sobre demanda{oferta_anual / demanda_anual:8.3f}")
print(f"confiabilidad con {VOLUMEN_BASE:.0f} m3 {confiabilidad_base:8.4f}")
print(f"déficit medio anual       {deficit_anual:8.1f} m3")
print(f"rebose medio anual        {reb[EVAL].sum() / (ANIOS - 1):8.1f} m3")

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(13.5 / 2.54, 10.0 / 2.54),
                         sharex=True, layout="constrained")
dias = np.arange(365)
ejes[0].bar(dias, lluvia.to_numpy()[365:730], color=PALETA["azul"], width=1.0)
ejes[0].set_ylabel("Lluvia diaria (mm)")
ejes[0].set_title("Segundo año de simulación con el tanque de 30 m3")
ejes[1].plot(dias, alm[365:730], color=PALETA["verde"], lw=1.3,
             label="volumen almacenado")
ejes[1].axhline(VOLUMEN_BASE, color=PALETA["gris"], lw=0.9, ls="--",
                label="capacidad del tanque")
falla = sumin[365:730] < DEMANDA - 1e-12
ejes[1].plot(dias[falla], np.zeros(int(falla.sum())), ".", ms=3,
             color=PALETA["rojo"], label="día sin servicio completo")
ejes[1].set_xlabel("Día del año")
ejes[1].set_ylabel("Volumen almacenado (m3)")
ejes[1].set_xlim(0, 364)
ejes[1].set_ylim(-1.5, VOLUMEN_BASE * 1.15)
ejes[1].legend(loc="upper left", fontsize=7)
plt.show()

### Comentario de la Tarea 4

**Lectura del resultado.** COMPLETE. Compare la razón entre la oferta anual y la demanda
anual con la confiabilidad obtenida y explique la discrepancia con la estacionalidad de la
lluvia.

## Tarea 5. Adimensionalización e invariancia de la respuesta

La Definición 1.12 del libro presenta la adimensionalización como el procedimiento que refiere
cada variable a una escala característica del propio problema, de modo que la solución quede
gobernada por un número reducido de grupos. En este modelo la escala natural de volumen es la
oferta media anual, que se obtiene del área de cubierta, del coeficiente de escorrentía y de la
lluvia anual.

Con esa escala aparecen dos grupos, la fracción de demanda y la fracción de almacenamiento.
Si el razonamiento es correcto, dos sistemas con los mismos grupos y la misma serie de lluvia
deben entregar exactamente la misma confiabilidad, aunque sus dimensiones difieran por un
factor arbitrario. Esa predicción es verificable y la celda de comprobación la pone a prueba.

In [ ]:
# Derivación simbólica de los dos grupos, con la escala de volumen del problema
A_s, C_s, P_s, V_s, D_s = sp.symbols("A C P_a V D", positive=True)
oferta_sim = C_s * A_s * P_s / 1000
frac_demanda_sim = sp.simplify(365 * D_s / oferta_sim)
frac_almacen_sim = sp.simplify(V_s / oferta_sim)
print("oferta anual característica ", oferta_sim)
print("fracción de demanda         ", frac_demanda_sim)
print("fracción de almacenamiento  ", frac_almacen_sim)

In [ ]:
# COMPLETE: evalúe los dos grupos adimensionales del caso base y la autonomía
#           del tanque en días, que es el volumen dividido por la demanda diaria.
frac_demanda = 0.0        # valores de partida evidentemente incorrectos
frac_almacenamiento = 0.0
autonomia_dias = 0.0

print(f"fracción de demanda        {frac_demanda:.4f}")
print(f"fracción de almacenamiento {frac_almacenamiento:.4f}")
print(f"autonomía del tanque       {autonomia_dias:.1f} días")

In [ ]:
def confiabilidad(volumen, demanda=DEMANDA, area=AREA_TECHO):
    """Fracción de días con servicio completo, descartando el año de calentamiento."""
    _, _, s, _ = balance_diario(lluvia.to_numpy(), area, COEF_ESCORRENTIA,
                                LAVADO_INICIAL, volumen, demanda)
    return float(np.mean(s[EVAL] >= demanda - 1e-12))


FACTOR = 2.7
c_base = confiabilidad(VOLUMEN_BASE)
c_escalado = confiabilidad(VOLUMEN_BASE * FACTOR, DEMANDA * FACTOR, AREA_TECHO * FACTOR)

escala = oferta_anual if oferta_anual > 0.0 else float("nan")
comprobar("la fracción de demanda coincide con la razón de volúmenes anuales",
          abs(frac_demanda - demanda_anual / escala) < 1e-9,
          f"grupo {frac_demanda:.4f}")
comprobar("la fracción de almacenamiento usa la oferta anual como escala",
          abs(frac_almacenamiento - VOLUMEN_BASE / escala) < 1e-9,
          f"grupo {frac_almacenamiento:.4f}")
comprobar("la autonomía del tanque está en días",
          abs(autonomia_dias - VOLUMEN_BASE / DEMANDA) < 1e-9,
          f"{autonomia_dias:.1f} días")
comprobar("la confiabilidad solo depende de los grupos adimensionales",
          abs(c_base - c_escalado) < 1e-12 and c_base > 0.0,
          f"{c_base:.6f} frente a {c_escalado:.6f} al escalar por {FACTOR}")

## Tarea 6. Curva de confiabilidad, decisión de diseño y dominio de validez

Con el modelo verificado ya es posible responder la pregunta de ingeniería. Calcule la
confiabilidad para una malla de volúmenes de tanque, interpole la curva y determine el volumen
que alcanza el 80 por ciento y el que alcanza el 90 por ciento de días con servicio completo.

La interpolación monótona de Pchip es adecuada aquí porque la confiabilidad crece con el
volumen y no debe oscilar entre los puntos calculados. La función `volumen_para` devuelve un
valor no numérico cuando el objetivo queda fuera del rango alcanzado, de manera que el
cuaderno no se detiene si el modelo todavía no está completo.

In [ ]:
def volumen_para(objetivo, volumenes, curva):
    """Volumen que alcanza la confiabilidad objetivo, por interpolación monótona."""
    if float(np.min(curva)) > objetivo or float(np.max(curva)) < objetivo:
        return float("nan")
    interp = PchipInterpolator(volumenes, curva)
    return float(brentq(lambda v: float(interp(v)) - objetivo,
                        float(volumenes[0]), float(volumenes[-1])))


volumenes = np.arange(5.0, 205.0, 5.0)

In [ ]:
# COMPLETE: evalúe la confiabilidad para cada volumen de la malla `volumenes`.
curva = np.zeros_like(volumenes)      # valor de partida evidentemente incorrecto

v80 = volumen_para(0.80, volumenes, curva)
v90 = volumen_para(0.90, volumenes, curva)
print(f"volumen para el 80 por ciento de confiabilidad  {v80:6.1f} m3")
print(f"volumen para el 90 por ciento de confiabilidad  {v90:6.1f} m3")

In [ ]:
creciente = bool(np.all(np.diff(curva) >= -1e-12))
comprobar("la confiabilidad no decrece al aumentar el volumen", creciente)
comprobar("la curva alcanza el 90 por ciento dentro de la malla ensayada",
          np.isfinite(v90), f"v90 = {v90}")
comprobar("el tanque del anteproyecto queda por debajo del 80 por ciento",
          float(curva[volumenes == VOLUMEN_BASE][0]) < 0.80 if np.any(volumenes == VOLUMEN_BASE)
          else False,
          f"confiabilidad con {VOLUMEN_BASE:.0f} m3 igual a "
          f"{float(curva[volumenes == VOLUMEN_BASE][0]):.4f}")
comprobar("el volumen para el 90 por ciento supera al del anteproyecto en más del doble",
          np.isfinite(v90) and v90 > 2 * VOLUMEN_BASE, f"v90 = {v90:.1f} m3")

In [ ]:
fig, eje = plt.subplots(figsize=(13.5 / 2.54, 7.2 / 2.54), layout="constrained")
eje.plot(volumenes, 100 * curva, color=PALETA["azul"], lw=1.5,
         label="modelo dinámico diario")
eje.axhline(80.0, color=PALETA["naranja"], lw=0.9, ls="--", label="meta del 80 por ciento")
eje.axhline(90.0, color=PALETA["rojo"], lw=0.9, ls=":", label="meta del 90 por ciento")
eje.axvline(VOLUMEN_BASE, color=PALETA["gris"], lw=0.9,
            label="tanque del anteproyecto")
eje.set_xlabel("Volumen del tanque (m3)")
eje.set_ylabel("Días con servicio completo (%)")
eje.set_xlim(0.0, 200.0)
eje.set_ylim(40.0, 102.0)
eje.legend(loc="lower right", fontsize=7)
plt.show()

### Comentario de la Tarea 6

**Decisión de diseño.** COMPLETE. Indique qué volumen recomienda, con qué confiabilidad, y
qué alternativa propone si el volumen necesario resulta desproporcionado.

**Dominio de validez.** COMPLETE. Delimite el conjunto de condiciones en las cuales la
respuesta del modelo merece crédito, siguiendo la Definición 1.13 del libro, y nombre al menos
tres razones por las cuales la confiabilidad calculada no debe reportarse como una predicción
para el año entrante.

## Cierre y lista de comprobación

Antes de entregar, confirme que puede responder afirmativamente a cada punto.

1. El cuaderno corre completo de arriba abajo con `REVISAR = True` tras reiniciar el núcleo.
2. Todas las verificaciones informan el estado correcto.
3. El registro de supuestos tiene al menos cinco filas y las cuatro columnas exigidas.
4. Sabe explicar por qué el balance anual y el balance diario dan respuestas distintas.
5. Sabe justificar por qué el cierre del balance de masa debe ser del orden de la precisión
   de la máquina y qué significaría un residuo mayor.
6. El repositorio incluye el archivo `entorno.json`, el archivo `README.md` y el archivo
   `DECLARACION.md` con el uso de asistentes de programación.

Si algo no salió, revise en el libro la Sección 1.1 para la distinción entre modelar y
simular, la Sección 1.2 y la Tabla 1.1 para la clasificación, la Sección 1.3 y el Listado 1.2
para el balance diario, la Sección 1.4 y el Listado 1.4 para los grados de libertad, la
Sección 1.5 para la adimensionalización y la Sección 1.6 para el dominio de validez. El
Problema 1-27 del capítulo plantea el mismo sistema como ejercicio abierto de diseño.